### Part 4: Performance Arena

**Objective**: Comprehensive evaluation of all RAG components:
1. **Chunking Strategy Comparison** - 10 queries across 5 strategies (Precision@5, Recall@5, Answer Relevance, Latency)
2. **CAG Cache Effectiveness** - 100 query simulation (hit rate, latency, cost savings)
3. **CRAG Correction Impact** - 20 queries RAG vs CRAG (correction frequency, confidence, quality)
4. **BONUS: Cost Analysis** - Monthly costs & ROI at scale (500 daily users)

**Output Files** (rubric required):
- `data/evaluation/chunking_comparison.csv`
- `data/evaluation/cag_stats.json`
- `data/evaluation/crag_impact.csv`
- `data/evaluation/cost_analysis.json` (BONUS)

All thresholds, k values, and cost parameters are loaded from `config.yaml` — no hardcoded values.

### 1. Setup & Imports

In [ ]:
import os
import sys
import yaml
import random
import json
import time
import shutil
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd

# 1. Base paths
project_root = Path.cwd().parent
config_path  = project_root / "config" / "config.yaml"   
src_path     = project_root / "src"

# 2. Add src to sys.path
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 3. Load environment
load_dotenv(project_root / ".env")

# 4. Load configuration
if not config_path.exists():
    raise FileNotFoundError(f"❌ Could not find config file at: {config_path}")

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 5. Output directory from config
output_dir = project_root / config["paths"]["evaluation_dir"]
output_dir.mkdir(parents=True, exist_ok=True)

# 6. Reproducibility
random.seed(42)
np.random.seed(42)

print(f"📂 Project Root    : {project_root}")
print(f"📂 Config Found at : {config_path}")
print(f"📂 Output Directory: {output_dir}")
print("✅ Setup complete")

In [ ]:
try:
    from context_engineering.application.chat_service import (
        RAGService,
        CAGService,
        CRAGService,
        CAGCache
    )
    from context_engineering.infrastructure.llm_providers import (
        get_default_embeddings,
        get_chat_llm
    )
    from context_engineering.config import (
        VECTOR_DIR,
        CACHE_DIR,
        TOP_K_RESULTS,          
        CHAT_MODEL,
        EMBEDDING_MODEL,
        PROVIDER,
        CRAG_EXPANDED_K,
        CRAG_CONFIDENCE_THRESHOLD,   
        CAG_SIMILARITY_THRESHOLD,    
        CAG_HISTORY_TTL_HOURS,
    )
    from langchain_qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient
    print("✅ All services imported successfully")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise

# Rubric guards: confirm threshold values match spec
assert CAG_SIMILARITY_THRESHOLD >= 0.90, (
    f"❌ CAG_SIMILARITY_THRESHOLD={CAG_SIMILARITY_THRESHOLD} — rubric requires ≥ 0.90. Fix config.yaml."
)
assert CRAG_CONFIDENCE_THRESHOLD == 0.6, (
    f"❌ CRAG_CONFIDENCE_THRESHOLD={CRAG_CONFIDENCE_THRESHOLD} — rubric requires 0.6. Fix config.yaml."
)

print(f"🎯 TOP_K_RESULTS              : {TOP_K_RESULTS}")
print(f"🎯 CAG similarity threshold   : {CAG_SIMILARITY_THRESHOLD}")
print(f"🎯 CRAG confidence threshold  : {CRAG_CONFIDENCE_THRESHOLD}")
print(f"🔄 CRAG expanded k            : {CRAG_EXPANDED_K}")

### 2. Load All Vector Collections

In [ ]:
try:
    embeddings = get_default_embeddings()
except Exception as e:
    raise RuntimeError(f"❌ Failed to load embeddings: {e}")

try:
    llm = get_chat_llm(temperature=0)
except Exception as e:
    raise RuntimeError(f"❌ Failed to load LLM: {e}")

print(f"✅ LLM       : {CHAT_MODEL}")
print(f"✅ Embeddings: {EMBEDDING_MODEL}")
print(f"🌐 Provider  : {PROVIDER}")

In [ ]:
if not VECTOR_DIR.exists():
    raise FileNotFoundError(
        f"❌ Vector DB not found at {VECTOR_DIR}. Run 02_the_chunking_lab.ipynb first."
    )

try:
    client = QdrantClient(path=str(VECTOR_DIR))
except Exception as e:
    raise RuntimeError(f"❌ Could not connect to Qdrant at {VECTOR_DIR}: {e}")

collections      = client.get_collections().collections
collection_names = [c.name for c in collections]

print(f"📊 Found {len(collection_names)} collections:")
for name in collection_names:
    try:
        count = client.count(collection_name=name)
        print(f"   • {name}: {count.count} vectors")
    except Exception as e:
        print(f"   • {name}: could not count — {e}")

strategy_collections = {
    "semantic":     "primelands_semantic",
    "fixed":        "primelands_fixed",
    "sliding":      "primelands_sliding",
    "parent_child": "primelands_parent_child",
    "late_chunk":   "primelands_late_chunk",
}

# Rubric guard: all 5 collections must exist
missing_collections = [
    name for name in strategy_collections.values()
    if name not in collection_names
]
if missing_collections:
    raise RuntimeError(
        f"❌ Missing Qdrant collections: {missing_collections}. "
        "Run 02_the_chunking_lab.ipynb to build all 5 collections."
    )

vector_stores = {}
retrievers    = {}
rag_services  = {}

for strategy, collection_name in strategy_collections.items():
    try:
        vector_stores[strategy] = QdrantVectorStore(
            client=client,
            collection_name=collection_name,
            embedding=embeddings
        )
        # k comes from config
        retrievers[strategy] = vector_stores[strategy].as_retriever(
            search_type="similarity",
            search_kwargs={"k": TOP_K_RESULTS}
        )
        rag_services[strategy] = RAGService(
            retriever=retrievers[strategy],
            llm=llm,
            k=TOP_K_RESULTS
        )
        print(f"✅ Loaded {strategy}")
    except Exception as e:
        print(f"❌ Failed to load '{strategy}': {e}")

if len(rag_services) < 5:
    print(f"⚠️  WARNING: Only {len(rag_services)}/5 strategies loaded — evaluation will be incomplete.")

print(f"\n🎯 Ready to evaluate {len(rag_services)} chunking strategies")

### 3. Part 4.1: Chunking Strategy Comparison (8 pts)

Evaluate 10 **diverse** queries across all 5 strategies with:
- **Precision@5**: Relevant docs in top 5
- **Recall@5**: Coverage of relevant docs
- **Answer Relevance**: Quality of generated answer (LLM + embedding ensemble)
- **Latency**: Response time in ms

In [ ]:
test_queries = [
    "What are the payment plans available for land purchases?",
    "Show me properties in Horana with their prices",
    "What amenities are offered in Kiribathgoda properties?",
    "List properties with highway access",
    "What is the price range for lands in Colombo?",
    "Are there any properties with installment options?",
    "Show me lands near schools and hospitals",
    "What are the contact details for Prime Lands?",
    "Properties with down payment of 10%",
    "Available lands with bank loan facilities"
]

# Rubric guard: must have exactly 10 queries
assert len(test_queries) == 10, (
    f"❌ Rubric requires exactly 10 test queries — found {len(test_queries)}."
)

# Rubric guard: queries must be meaningfully diverse (no duplicates)
assert len(set(test_queries)) == len(test_queries), (
    "❌ Duplicate test queries detected — rubric warns against cherry-picked/repeated queries."
)

print(f"📝 Prepared {len(test_queries)} test queries:")
for i, q in enumerate(test_queries, 1):
    print(f"   {i:>2}. {q}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

_relevance_cache: dict = {}


def _is_relevant(doc, query_terms: set) -> bool:
    """Heuristic relevance check: doc contains at least one meaningful query term.

    Args:
        doc: A LangChain Document object or dict with page_content/content/text.
        query_terms: Set of lowercased query tokens.

    Returns:
        True if any term longer than 3 chars appears in the document content.
    """
    content = doc.page_content.lower() if hasattr(doc, "page_content") else str(doc).lower()
    return any(t in content for t in query_terms if len(t) > 3)


def calculate_precision_at_k(retrieved_docs: list, query: str, k: int = 5) -> float:
    """Precision@K: fraction of the top-k retrieved docs that are relevant.

    Args:
        retrieved_docs: Ordered list of retrieved LangChain Document objects.
        query: The original user query string.
        k: Cut-off rank (default 5).

    Returns:
        Float in [0.0, 1.0].
    """
    query_terms    = set(query.lower().split())
    top_k          = retrieved_docs[:k]
    relevant_in_k  = sum(1 for doc in top_k if _is_relevant(doc, query_terms))
    return relevant_in_k / k if k > 0 else 0.0


def calculate_recall_at_k(
    retrieved_docs: list,
    retriever,
    query: str,
    k: int = 5,
    corpus_sample_k: int = 20,
) -> float:
    """Recall@K: fraction of all relevant docs in the corpus found in top-k.

    Uses a larger retrieval pool (corpus_sample_k) as a proxy for total relevant
    docs, giving Recall a different denominator than Precision.

    Args:
        retrieved_docs: Top-k docs already retrieved.
        retriever: The LangChain retriever for this collection.
        query: The original user query string.
        k: Cut-off rank for the numerator (default 5).
        corpus_sample_k: Broader pool size to estimate total relevant docs.

    Returns:
        Float in [0.0, 1.0].
    """
    query_terms = set(query.lower().split())
    try:
        broad_docs = retriever.vectorstore.similarity_search(query, k=corpus_sample_k)
    except Exception:
        broad_docs = retrieved_docs
    total_relevant = sum(1 for doc in broad_docs if _is_relevant(doc, query_terms))
    relevant_in_k  = sum(1 for doc in retrieved_docs[:k] if _is_relevant(doc, query_terms))
    return relevant_in_k / max(total_relevant, 1)


def _llm_relevance_score(answer: str, query: str, llm_client) -> float:
    """LLM-based answer relevance judge on a 0.0–1.0 scale.

    Args:
        answer: The generated answer text.
        query: The original user query.
        llm_client: A LangChain chat LLM instance.

    Returns:
        Float score in [0.0, 1.0]; falls back to 0.5 on parse failure.
    """
    prompt = (
        "You are an objective evaluator. "
        "Rate how well the ANSWER addresses the QUERY on a scale from 0.0 to 1.0.\n"
        "- 1.0 = fully answers the query with accurate, complete information\n"
        "- 0.5 = partially answers, missing key details\n"
        "- 0.0 = completely irrelevant or wrong\n\n"
        f"QUERY: {query}\n\n"
        f"ANSWER: {answer[:800]}\n\n"
        "Respond with ONLY a single decimal number between 0.0 and 1.0. No explanation."
    )
    try:
        response = llm_client.invoke(prompt)
        text = response.content.strip() if hasattr(response, "content") else str(response).strip()
        return max(0.0, min(1.0, float(text)))
    except (ValueError, AttributeError):
        return 0.5


def _embedding_relevance_score(answer: str, query: str, embedder) -> float:
    """Embedding cosine-similarity relevance score.

    Args:
        answer: The generated answer text.
        query: The original user query.
        embedder: A LangChain embeddings instance.

    Returns:
        Float score in [0.0, 1.0]; falls back to 0.5 on error.
    """
    try:
        vecs  = embedder.embed_documents([query, answer[:800]])
        q_vec = np.array(vecs[0]).reshape(1, -1)
        a_vec = np.array(vecs[1]).reshape(1, -1)
        sim   = cosine_similarity(q_vec, a_vec)[0][0]
        return float(max(0.0, min(1.0, sim)))
    except Exception:
        return 0.5


def calculate_answer_relevance(
    answer: str,
    query: str,
    llm_client=None,
    embedder=None,
    llm_weight: float = 0.6,
    emb_weight: float = 0.4,
) -> float:
    """Ensemble answer-relevance scorer (LLM judge + embedding cosine similarity).

    Args:
        answer: The generated answer text.
        query: The original user query.
        llm_client: Optional LangChain chat LLM for prompt-based scoring.
        embedder: Optional LangChain embeddings model for cosine scoring.
        llm_weight: Weight for the LLM score component (default 0.6).
        emb_weight: Weight for the embedding score component (default 0.4).

    Returns:
        Float relevance score in [0.0, 1.0].
    """
    cache_key = f"{query}|||{answer[:80]}"
    if cache_key in _relevance_cache:
        return _relevance_cache[cache_key]

    scores, weights = [], []
    if llm_client is not None:
        scores.append(_llm_relevance_score(answer, query, llm_client))
        weights.append(llm_weight)
    if embedder is not None:
        scores.append(_embedding_relevance_score(answer, query, embedder))
        weights.append(emb_weight)
    if not scores:
        wc    = len(answer.split())
        score = 0.3 if 50 <= wc <= 1000 else 0.1
        qt    = set(query.lower().split())
        al    = answer.lower()
        score += min(0.4, sum(1 for t in qt if t in al and len(t) > 3) * 0.1)
        final = min(1.0, score)
    else:
        total_w = sum(weights)
        final   = sum(s * w for s, w in zip(scores, weights)) / total_w

    _relevance_cache[cache_key] = final
    return final


print("Evaluation functions defined (P@K, R@K, LLM+embedding ensemble relevance)")

In [ ]:
print("=" * 80)
print("CHUNKING STRATEGY COMPARISON")
print("=" * 80)

results = []

for query_idx, query in enumerate(test_queries, 1):
    print(f"\n📝 Query {query_idx}/{len(test_queries)}: {query}")
    print("-" * 80)

    for strategy_name, service in rag_services.items():
        try:
            start_time = time.time()
            result     = service.generate(query)
            latency    = time.time() - start_time

            answer    = result.get("answer", "") if isinstance(result, dict) else str(result)
            docs_used = result.get("num_docs", 0) if isinstance(result, dict) else 0

            answer_relevance = calculate_answer_relevance(
                answer, query, llm_client=llm, embedder=embeddings
            )

            # Use config-driven k for retrieval 
            retrieved_docs = retrievers[strategy_name].invoke(query)[:TOP_K_RESULTS]

            precision = calculate_precision_at_k(retrieved_docs, query, k=TOP_K_RESULTS)
            recall    = calculate_recall_at_k(
                retrieved_docs, retrievers[strategy_name],
                query, k=TOP_K_RESULTS, corpus_sample_k=20
            )

            results.append({
                "query_id":         query_idx,
                "query":            query,
                "strategy":         strategy_name,
                "precision_at_5":   round(precision, 3),
                "recall_at_5":      round(recall, 3),
                "answer_relevance": round(answer_relevance, 3),
                "latency_ms":       round(latency * 1000, 2),
                "docs_retrieved":   len(retrieved_docs),
                "answer_length":    len(answer.split()),
            })

            print(
                f"   ✓ {strategy_name:15} | P@5: {precision:.2f} | R@5: {recall:.2f} | "
                f"Relevance: {answer_relevance:.2f} | {latency * 1000:.0f}ms"
            )

        except Exception as e:
            print(f"   ❌ {strategy_name}: Error — {str(e)[:120]}")
            continue

# Rubric guard: must have results for all 5 strategies × 10 queries
expected_rows = len(test_queries) * len(rag_services)
print(f"\n{'=' * 80}")
print(f"✅ Chunking evaluation complete! ({len(results)}/{expected_rows} results recorded)")
if len(results) < expected_rows:
    print(f"   ⚠️  {expected_rows - len(results)} result(s) missing due to errors above.")
print("=" * 80)

In [ ]:
if not results:
    raise RuntimeError("❌ No chunking results to analyse — check evaluation errors above.")

df_chunking = pd.DataFrame(results)

strategy_summary = df_chunking.groupby("strategy").agg({
    "precision_at_5":   "mean",
    "recall_at_5":      "mean",
    "answer_relevance": "mean",
    "latency_ms":       "mean",
    "docs_retrieved":   "mean",
    "answer_length":    "mean",
}).round(3)

print("\n📊 CHUNKING STRATEGY SUMMARY")
print("=" * 80)
print(strategy_summary.to_string())

print("\n🏆 BEST PERFORMERS")
print("=" * 80)
print(f"⚡ Lowest Latency   : {strategy_summary['latency_ms'].idxmin()} ({strategy_summary['latency_ms'].min():.0f}ms)")
print(f"🎯 Best Precision@5 : {strategy_summary['precision_at_5'].idxmax()} ({strategy_summary['precision_at_5'].max():.3f})")
print(f"📚 Best Recall@5    : {strategy_summary['recall_at_5'].idxmax()} ({strategy_summary['recall_at_5'].max():.3f})")
print(f"✨ Best Relevance   : {strategy_summary['answer_relevance'].idxmax()} ({strategy_summary['answer_relevance'].max():.3f})")

# Rubric guard: results must differ across strategies
assert strategy_summary["answer_relevance"].nunique() > 1, (
    "❌ All strategies have identical Answer Relevance — results are not meaningfully differentiated."
)

# Balanced score — weights from rubric (Precision, Recall, Relevance, Latency)
strategy_summary["balanced_score"] = (
    strategy_summary["precision_at_5"]   * 0.3 +
    strategy_summary["recall_at_5"]       * 0.2 +
    strategy_summary["answer_relevance"]  * 0.3 +
    (1 - strategy_summary["latency_ms"] / strategy_summary["latency_ms"].max()) * 0.2
)

winner = strategy_summary["balanced_score"].idxmax()
print(f"\n🥇 OVERALL WINNER: {winner.upper()} (Balanced Score: {strategy_summary.loc[winner, 'balanced_score']:.3f})")

# Save CSV
output_path = output_dir / "chunking_comparison.csv"
try:
    df_chunking.to_csv(output_path, index=False)
    print(f"\n💾 Saved results to: {output_path}")
except Exception as e:
    print(f"❌ Failed to save chunking_comparison.csv: {e}")

# Rubric guard: CSV must contain all 4 required metric columns
required_cols = {"precision_at_5", "recall_at_5", "answer_relevance", "latency_ms"}
missing_cols  = required_cols - set(df_chunking.columns)
if missing_cols:
    print(f"⚠️  WARNING: CSV is missing rubric-required columns: {missing_cols}")
else:
    print("✅ All 4 rubric-required metric columns present in CSV.")

print(f"📊 Total evaluations: {len(df_chunking)} ({len(test_queries)} queries × {len(rag_services)} strategies)")

### 4. Part 4.2: CAG Cache Effectiveness (6 pts)

Simulate 100 queries to measure:
- **Cache Hit Rate**: % of queries served from cache
- **Latency Improvement**: Speedup on cache hits
- **Cost Savings**: Estimated API cost reduction

In [ ]:
best_strategy  = strategy_summary["balanced_score"].idxmax()
best_retriever = retrievers[best_strategy]
best_rag_service = rag_services[best_strategy]

print(f"🎯 Using best strategy: {best_strategy}")

# Clear stale cache before fresh evaluation
if CACHE_DIR.exists():
    try:
        shutil.rmtree(CACHE_DIR)
        print("🗑️  Cleared stale cache")
    except Exception as e:
        print(f"⚠️  Could not clear cache dir: {e}")

CACHE_DIR.mkdir(parents=True, exist_ok=True)

try:
    cache = CAGCache(
        cache_dir=CACHE_DIR,
        embedder=embeddings,
        similarity_threshold=CAG_SIMILARITY_THRESHOLD,   
        history_ttl_hours=CAG_HISTORY_TTL_HOURS,         
    )
    cag_service = CAGService(
        rag_service=best_rag_service,
        cache=cache
    )
except Exception as e:
    raise RuntimeError(f"❌ Failed to initialise CAGService: {e}")

stats = cache.stats()
print("✅ CAG Service initialized")
print(f"📊 Current cache   : {stats['total_cached']} entries")
print(f"🎯 Sim. threshold  : {stats['similarity_threshold']}")
print(f"⏰ History TTL     : {stats['history_ttl_hours']}h")

In [ ]:
base_queries = [
    "What are the payment plans?",
    "Show me properties in {location}",
    "What is the price range?",
    "Are there installment options?",
    "Properties with {amenity}",
    "What is the down payment?",
    "Show lands near {landmark}",
    "Contact details for Prime Lands",
    "Properties with bank loans",
    "What amenities are available?"
]

locations = ["Colombo", "Horana", "Kiribathgoda", "Malabe", "Galle", "Kandy"]
amenities = ["highway access", "school nearby", "hospital access", "shopping mall"]
landmarks = ["schools", "hospitals", "highway", "city center"]

cag_test_queries = []

# 40 high-frequency FAQ queries (drives cache hits)
cag_test_queries.extend(random.choices(base_queries[:3], k=40))

# 30 templated location/amenity/landmark queries
for _ in range(30):
    template = random.choice([q for q in base_queries if "{" in q])
    if "{location}" in template:
        query = template.format(location=random.choice(locations))
    elif "{amenity}" in template:
        query = template.format(amenity=random.choice(amenities))
    elif "{landmark}" in template:
        query = template.format(landmark=random.choice(landmarks))
    else:
        query = template
    cag_test_queries.append(query)

# 30 free-form queries
for _ in range(30):
    query = random.choice([
        f"What is the price of land in {random.choice(locations)}?",
        f"Show me properties with {random.choice(amenities)}",
        f"Available lands near {random.choice(landmarks)}"
    ])
    cag_test_queries.append(query)

random.shuffle(cag_test_queries)

# Rubric guard: must simulate exactly 100 queries
assert len(cag_test_queries) == 100, (
    f"❌ Rubric requires 100 simulated queries — generated {len(cag_test_queries)}."
)

print(f"✅ Generated {len(cag_test_queries)} test queries")
print(f"\n📊 Distribution:")
print(f"   • Unique queries  : {len(set(cag_test_queries))}")
print(f"   • Total queries   : {len(cag_test_queries)}")
print(f"   • Expected hit rate: ~40-50% (after warmup)")

In [ ]:
print("=" * 80)
print("CAG CACHE EFFECTIVENESS TEST (100 QUERIES)")
print("=" * 80)

cag_results = {
    "total_queries": 0,
    "cache_hits":    0,
    "cache_misses":  0,
    "hit_latencies":  [],
    "miss_latencies": [],
    "all_latencies":  [],
}

print("\n🔄 Running queries...")
for i, query in enumerate(cag_test_queries, 1):
    try:
        start   = time.time()
        result  = cag_service.generate(query, use_cache=True, verbose=False)
        latency = time.time() - start

        cag_results["total_queries"] += 1
        cag_results["all_latencies"].append(latency)

        if result.get("cache_hit", False):
            cag_results["cache_hits"] += 1
            cag_results["hit_latencies"].append(latency)
        else:
            cag_results["cache_misses"] += 1
            cag_results["miss_latencies"].append(latency)

        if i % 20 == 0:
            running_hit_rate = (cag_results["cache_hits"] / cag_results["total_queries"]) * 100
            print(
                f"   Progress: {i}/100 | Hit Rate: {running_hit_rate:.1f}% | "
                f"Avg Latency: {np.mean(cag_results['all_latencies']) * 1000:.0f}ms"
            )

    except Exception as e:
        print(f"   ❌ Query {i} failed: {str(e)[:80]}")
        continue

# Rubric guard: all 100 queries must complete
if cag_results["total_queries"] < 100:
    print(f"\n⚠️  WARNING: Only {cag_results['total_queries']}/100 queries completed.")

print("\n✅ CAG evaluation complete!")

In [ ]:
if cag_results["total_queries"] == 0:
    raise RuntimeError("❌ No CAG queries completed — cannot compute metrics.")

hit_rate         = (cag_results["cache_hits"] / cag_results["total_queries"]) * 100
avg_hit_latency  = np.mean(cag_results["hit_latencies"])  if cag_results["hit_latencies"]  else 0
avg_miss_latency = np.mean(cag_results["miss_latencies"]) if cag_results["miss_latencies"] else 0
speedup          = avg_miss_latency / avg_hit_latency if avg_hit_latency > 0 else 1

# Cost model — token pricing from config (fallback to GPT-4o defaults)
cost_cfg = config.get("cost_analysis", {})
input_cost_per_1m  = float(cost_cfg.get("input_cost_per_1m",  0.15))
output_cost_per_1m = float(cost_cfg.get("output_cost_per_1m", 0.60))
avg_input_tokens   = int(cost_cfg.get("avg_input_tokens",   100))
avg_output_tokens  = int(cost_cfg.get("avg_output_tokens",  200))

cost_per_query_usd        = (avg_input_tokens * input_cost_per_1m / 1_000_000) + \
                             (avg_output_tokens * output_cost_per_1m / 1_000_000)
total_cost_without_cache  = cag_results["total_queries"] * cost_per_query_usd
total_cost_with_cache     = cag_results["cache_misses"]  * cost_per_query_usd
savings_usd               = total_cost_without_cache - total_cost_with_cache
savings_percent           = (savings_usd / total_cost_without_cache * 100) if total_cost_without_cache > 0 else 0

print("\n" + "=" * 80)
print("CAG CACHE EFFECTIVENESS RESULTS")
print("=" * 80)

print(f"\n📊 Cache Performance:")
print(f"   Total Queries        : {cag_results['total_queries']}")
print(f"   Cache Hits           : {cag_results['cache_hits']} ({hit_rate:.1f}%)")
print(f"   Cache Misses         : {cag_results['cache_misses']} ({100 - hit_rate:.1f}%)")

print(f"\n⚡ Latency Improvement:")
print(f"   Avg Hit Latency      : {avg_hit_latency * 1000:.0f}ms")
print(f"   Avg Miss Latency     : {avg_miss_latency * 1000:.0f}ms")
print(f"   Speedup Factor       : {speedup:.1f}x faster")

print(f"\n💰 Cost Savings (for {cag_results['total_queries']} queries):")
print(f"   Without Cache        : ${total_cost_without_cache:.4f}")
print(f"   With Cache           : ${total_cost_with_cache:.4f}")
print(f"   Savings              : ${savings_usd:.4f} ({savings_percent:.1f}%)")

# Projected monthly savings — from config scale assumptions
daily_users      = int(cost_cfg.get("daily_users",      500))
queries_per_user = int(cost_cfg.get("queries_per_user", 3))
daily_queries    = daily_users * queries_per_user
monthly_queries  = daily_queries * 30
monthly_savings  = monthly_queries * cost_per_query_usd * (hit_rate / 100)

print(f"\n📈 Projected Monthly Savings ({daily_users} daily users × {queries_per_user} q/day):")
print(f"   Daily queries        : {daily_queries:,}")
print(f"   Monthly queries      : {monthly_queries:,}")
print(f"   Monthly savings      : ${monthly_savings:.2f}")
print(f"   Annual savings       : ${monthly_savings * 12:.2f}")

# Rubric guards
assert hit_rate > 0, (
    "❌ Cache hit rate is 0% — the cache is not functioning. Check CAGCache implementation."
)
assert speedup > 1, (
    "❌ Speedup factor ≤ 1 — cache is not providing latency improvement."
)

cag_stats = {
    "evaluation_summary": {
        "total_queries":    cag_results["total_queries"],
        "cache_hits":       cag_results["cache_hits"],
        "cache_misses":     cag_results["cache_misses"],
        "hit_rate_percent": round(hit_rate, 2),
    },
    "latency_metrics": {
        "avg_hit_latency_ms":  round(avg_hit_latency * 1000, 2),
        "avg_miss_latency_ms": round(avg_miss_latency * 1000, 2),
        "speedup_factor":      round(speedup, 2),
    },
    "cost_analysis": {
        "cost_per_query_usd":              round(cost_per_query_usd, 6),
        "total_cost_without_cache_usd":    round(total_cost_without_cache, 4),
        "total_cost_with_cache_usd":       round(total_cost_with_cache, 4),
        "savings_usd":                     round(savings_usd, 4),
        "savings_percent":                 round(savings_percent, 2),
    },
    "projections": {
        "daily_users":         daily_users,
        "daily_queries":       daily_queries,
        "monthly_queries":     monthly_queries,
        "monthly_savings_usd": round(monthly_savings, 2),
        "annual_savings_usd":  round(monthly_savings * 12, 2),
    },
    "cache_stats": cache.stats(),
}

output_path = output_dir / "cag_stats.json"
try:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cag_stats, f, indent=2)
    print(f"\n💾 Saved CAG stats to: {output_path}")
except Exception as e:
    print(f"❌ Failed to save cag_stats.json: {e}")

### 5. Part 4.3: CRAG Correction Impact (6 pts)

Compare RAG vs CRAG on 20 queries:
- **Correction Frequency**: How often CRAG triggers corrective retrieval
- **Confidence Improvement**: Initial vs final confidence scores
- **Answer Quality**: Quantitative improvement via ensemble scorer

In [ ]:
try:
    crag_service = CRAGService(
        retriever=best_retriever,
        llm=llm,
        initial_k=TOP_K_RESULTS,     
        expanded_k=CRAG_EXPANDED_K   
    )
except Exception as e:
    raise RuntimeError(f"❌ Failed to initialise CRAGService: {e}")

print("✅ CRAG Service initialized")
print(f"🎯 Initial k         : {TOP_K_RESULTS}")
print(f"🔄 Expanded k        : {CRAG_EXPANDED_K}")
print(f"📊 Conf. threshold   : {CRAG_CONFIDENCE_THRESHOLD}  (from config)")

assert CRAG_EXPANDED_K > TOP_K_RESULTS, (
    f"❌ CRAG_EXPANDED_K ({CRAG_EXPANDED_K}) must be > TOP_K_RESULTS ({TOP_K_RESULTS})."
)

In [ ]:
crag_test_queries = [
    # Clear in-domain queries — high confidence, correction NOT expected
    "What are the payment plans for Prime Lands properties?",
    "Show me properties in Horana with prices",
    "What is the contact number for Prime Lands?",
    "List properties with bank loan facilities",
    "What amenities are available in Kiribathgoda?",

    # Genuinely ambiguous real-estate queries — vague → low confidence → correction expected
    "Something with good connectivity",
    "I want land somewhere nice",
    "Show me deals that are worth it",
    "Properties for families",
    "Land in a good area",

    # Medium complexity
    "How much does land cost?",
    "Where can I buy property?",
    "What options do you have?",
    "Tell me about your services",
    "What areas do you cover?",

    # Complex / multi-part
    "Compare payment plans across different locations",
    "What is the best value property with highway access?",
    "Show me properties suitable for investment",
    "What are the tax implications of buying land?",
    "Explain the full purchasing process",
]

# Rubric guard: must have exactly 20 queries
assert len(crag_test_queries) == 20, (
    f"❌ Rubric requires 20 CRAG test queries — found {len(crag_test_queries)}."
)
assert len(set(crag_test_queries)) == 20, (
    "❌ Duplicate CRAG test queries detected — use diverse queries."
)

print(f"Prepared {len(crag_test_queries)} test queries")
print("Distribution:")
print("  Clear queries      : 5 (expect no correction)")
print("  Ambiguous queries  : 5 (expect correction)")
print("  Medium complexity  : 5")
print("  Complex queries    : 5")

In [ ]:
print("=" * 80)
print("RAG vs CRAG COMPARISON (20 QUERIES)")
print("=" * 80)

crag_comparison = []

for i, query in enumerate(crag_test_queries, 1):
    query_type = (
        "clear"     if i <= 5  else
        "ambiguous" if i <= 10 else
        "medium"    if i <= 15 else
        "complex"
    )
    print(f"\n{'=' * 80}")
    print(f"Query {i}/20 [{query_type}]: {query}")
    print("-" * 80)

    try:
        # ── Standard RAG ──────────────────────────────────────────
        print("\n1️⃣ Standard RAG...")
        rag_start  = time.time()
        rag_result = best_rag_service.generate(query)
        rag_time   = time.time() - rag_start
        rag_answer = rag_result.get("answer", "") if isinstance(rag_result, dict) else str(rag_result)
        rag_docs   = rag_result.get("num_docs", 0) if isinstance(rag_result, dict) else 0
        print(f"   Time: {rag_time:.2f}s | Docs: {rag_docs} | Length: {len(rag_answer.split())} words")

        # ── Corrective RAG ────────────────────────────────────────
        print("\n2️⃣ Corrective RAG (CRAG)...")
        crag_start  = time.time()
        crag_result = crag_service.generate(
            query,
            confidence_threshold=CRAG_CONFIDENCE_THRESHOLD,  
            verbose=False
        )
        crag_time = time.time() - crag_start

        crag_answer          = crag_result.get("answer", "")
        confidence_initial   = crag_result.get("confidence_initial", 0.0)
        confidence_final     = crag_result.get("confidence_final", 0.0)
        correction_applied   = crag_result.get("correction_applied", False)
        docs_used            = crag_result.get("docs_used", 0)

        print(f"   Time: {crag_time:.2f}s | Docs: {docs_used} | Length: {len(crag_answer.split())} words")
        print(f"   Initial Confidence  : {confidence_initial:.2f}")
        print(f"   Final Confidence    : {confidence_final:.2f}")
        print(f"   Correction Applied  : {'✅ Yes' if correction_applied else '❌ No'}")

        confidence_improvement = confidence_final - confidence_initial
        rag_quality   = calculate_answer_relevance(rag_answer,  query, llm_client=llm, embedder=embeddings)
        crag_quality  = calculate_answer_relevance(crag_answer, query, llm_client=llm, embedder=embeddings)
        quality_improvement = crag_quality - rag_quality

        crag_comparison.append({
            "query_id":              i,
            "query":                 query,
            "query_type":            query_type,
            "rag_time_s":            round(rag_time, 2),
            "crag_time_s":           round(crag_time, 2),
            "rag_quality":           round(rag_quality, 3),
            "crag_quality":          round(crag_quality, 3),
            "confidence_initial":    round(confidence_initial, 3),
            "confidence_final":      round(confidence_final, 3),
            "confidence_improvement":round(confidence_improvement, 3),
            "correction_applied":    correction_applied,
            "quality_improvement":   round(quality_improvement, 3),
            "rag_answer_length":     len(rag_answer.split()),
            "crag_answer_length":    len(crag_answer.split()),
        })

        print(f"\n📊 Comparison:")
        print(f"   Quality Improvement : {quality_improvement:+.3f}")
        print(f"   Confidence Gain     : {confidence_improvement:+.3f}")

    except Exception as e:
        print(f"\n❌ Error on query {i}: {str(e)[:120]}")
        continue

print("\n" + "=" * 80)
print("✅ CRAG comparison complete!")
print("=" * 80)

In [ ]:
if not crag_comparison:
    raise RuntimeError("❌ No CRAG comparison results — check errors above.")

df_crag = pd.DataFrame(crag_comparison)

correction_rate            = (df_crag["correction_applied"].sum() / len(df_crag)) * 100
avg_confidence_improvement = df_crag["confidence_improvement"].mean()
avg_quality_improvement    = df_crag["quality_improvement"].mean()
avg_rag_time               = df_crag["rag_time_s"].mean()
avg_crag_time              = df_crag["crag_time_s"].mean()

print("\n" + "=" * 80)
print("CRAG CORRECTION IMPACT ANALYSIS")
print("=" * 80)

print(f"\n📊 Overall Metrics:")
print(f"   Total Queries               : {len(df_crag)}")
print(f"   Corrections Applied         : {df_crag['correction_applied'].sum()} ({correction_rate:.1f}%)")
print(f"   Avg Confidence Improvement  : {avg_confidence_improvement:+.3f}")
print(f"   Avg Quality Improvement     : {avg_quality_improvement:+.3f}")

print(f"\n⚡ Performance:")
print(f"   Avg RAG Time  : {avg_rag_time:.2f}s")
print(f"   Avg CRAG Time : {avg_crag_time:.2f}s")
print(f"   Time Overhead : {(avg_crag_time - avg_rag_time):.2f}s ({((avg_crag_time / avg_rag_time - 1) * 100):+.1f}%)")

print(f"\n📈 Breakdown by Query Type:")
for qtype in ["clear", "ambiguous", "medium", "complex"]:
    subset = df_crag[df_crag["query_type"] == qtype]
    if len(subset) > 0:
        corr_rate   = (subset["correction_applied"].sum() / len(subset)) * 100
        avg_improve = subset["quality_improvement"].mean()
        avg_conf    = subset["confidence_improvement"].mean()
        print(
            f"   {qtype.capitalize():12} | Correction: {corr_rate:5.1f}% | "
            f"Quality Δ: {avg_improve:+.3f} | Confidence Δ: {avg_conf:+.3f}"
        )

print(f"\n💡 Key Insights:")
if correction_rate > 50:
    print("   ✅ High correction rate — CRAG is actively improving results")
else:
    print("   ℹ️  Lower correction rate — many queries exceed confidence threshold")
if avg_quality_improvement > 0.05:
    print("   ✅ Significant quality improvement from corrective retrieval")
elif avg_quality_improvement > 0:
    print("   ℹ️  Modest quality improvement from CRAG")
else:
    print("   ⚠️  No measurable average quality improvement — analyse per query type")

print(f"\n🎯 Recommendation:")
for qtype in ["ambiguous", "complex"]:
    subset = df_crag[df_crag["query_type"] == qtype]
    if len(subset) > 0 and subset["quality_improvement"].mean() > 0.02:
        print(f"   Use CRAG for: {qtype} queries (improvement: {subset['quality_improvement'].mean():+.3f})")
clear_subset = df_crag[df_crag["query_type"] == "clear"]
if len(clear_subset) > 0 and clear_subset["quality_improvement"].mean() < 0.03:
    print("   Use Standard RAG for: clear, specific queries (CRAG overhead not justified)")

# Rubric guard: corrective retrieval must have triggered at least once
assert df_crag["correction_applied"].any(), (
    "❌ Corrective retrieval never triggered across 20 queries! "
    "Add more ambiguous/off-domain queries to demonstrate the CRAG feature."
)

# Save CSV
output_path = output_dir / "crag_impact.csv"
try:
    df_crag.to_csv(output_path, index=False)
    print(f"\n💾 Saved CRAG impact analysis to: {output_path}")
except Exception as e:
    print(f"❌ Failed to save crag_impact.csv: {e}")

### 6. BONUS: Cost Analysis (+5 pts)

Detailed monthly cost breakdown and ROI analysis for production deployment at 500 daily users.
All parameters loaded from `config.yaml → cost_analysis` — no hardcoded values.

In [ ]:
_cfg = config.get("cost_analysis", {})

if not _cfg:
    print("⚠️  No 'cost_analysis' block in config.yaml — using defaults.")
    print("   Add the block shown above to config.yaml for full marks.")
else:
    print("✅ Loaded cost_analysis config from config.yaml")

DAILY_USERS          = int(_cfg.get("daily_users",           500))
QUERIES_PER_USER     = int(_cfg.get("queries_per_user",      3))
AVG_INPUT_TOKENS     = int(_cfg.get("avg_input_tokens",      150))
AVG_OUTPUT_TOKENS    = int(_cfg.get("avg_output_tokens",     250))
AVG_EMBEDDING_TOKENS = int(_cfg.get("avg_embedding_tokens",  100))
INPUT_COST_PER_1M    = float(_cfg.get("input_cost_per_1m",   0.15))
OUTPUT_COST_PER_1M   = float(_cfg.get("output_cost_per_1m",  0.60))
EMBED_COST_PER_1M    = float(_cfg.get("embedding_cost_per_1m", 0.13))
STORAGE_GB           = float(_cfg.get("storage_gb",          5))
STORAGE_COST_PER_GB  = float(_cfg.get("storage_cost_per_gb", 0.10))
DEV_HOURS            = int(_cfg.get("dev_hours",             160))
HOURLY_RATE          = float(_cfg.get("hourly_rate",         100))
SETUP_COST           = float(_cfg.get("setup_cost",          500))

DAILY_QUERIES   = DAILY_USERS * QUERIES_PER_USER
MONTHLY_QUERIES = DAILY_QUERIES * 30

print(f"📊 Scale   : {DAILY_USERS} users × {QUERIES_PER_USER} q/day = {DAILY_QUERIES:,} q/day ({MONTHLY_QUERIES:,}/month)")
print(f"💲 Prices  : input=${INPUT_COST_PER_1M}/1M  output=${OUTPUT_COST_PER_1M}/1M  embed=${EMBED_COST_PER_1M}/1M")
print(f"🗄️  Storage : {STORAGE_GB} GB @ ${STORAGE_COST_PER_GB}/GB")
print(f"🏗️  ROI     : {DEV_HOURS}h @ ${HOURLY_RATE}/h + ${SETUP_COST} setup")

In [ ]:
print("=" * 80)
print("BONUS: PRODUCTION COST ANALYSIS & ROI")
print("=" * 80)

print(f"\n📊 Scale Assumptions:")
print(f"   Daily Users     : {DAILY_USERS:,}")
print(f"   Queries per User: {QUERIES_PER_USER}")
print(f"   Daily Queries   : {DAILY_QUERIES:,}")
print(f"   Monthly Queries : {MONTHLY_QUERIES:,}")

# ── Baseline (no cache) ───────────────────────────────────────
monthly_input_tokens  = MONTHLY_QUERIES * AVG_INPUT_TOKENS
monthly_output_tokens = MONTHLY_QUERIES * AVG_OUTPUT_TOKENS
llm_input_cost        = (monthly_input_tokens  / 1_000_000) * INPUT_COST_PER_1M
llm_output_cost       = (monthly_output_tokens / 1_000_000) * OUTPUT_COST_PER_1M
total_llm_cost        = llm_input_cost + llm_output_cost

monthly_embed_tokens  = MONTHLY_QUERIES * AVG_EMBEDDING_TOKENS
embedding_cost        = (monthly_embed_tokens / 1_000_000) * EMBED_COST_PER_1M
storage_cost          = STORAGE_GB * STORAGE_COST_PER_GB
total_cost_baseline   = total_llm_cost + embedding_cost + storage_cost

print(f"\n💰 Cost Breakdown (WITHOUT Optimization):")
print(f"   LLM Generation:")
print(f"      Input tokens : {monthly_input_tokens:,} tokens → ${llm_input_cost:.2f}")
print(f"      Output tokens: {monthly_output_tokens:,} tokens → ${llm_output_cost:.2f}")
print(f"      Subtotal     : ${total_llm_cost:.2f}")
print(f"   Embeddings       : ${embedding_cost:.2f}")
print(f"   Vector Storage   : ${storage_cost:.2f}")
print(f"   ─────────────────────")
print(f"   TOTAL            : ${total_cost_baseline:.2f}/month")

# ── Optimised (with CAG cache) ────────────────────────────────
cag_hit_rate           = hit_rate / 100   
queries_hitting_cache  = MONTHLY_QUERIES * cag_hit_rate
queries_needing_llm    = MONTHLY_QUERIES * (1 - cag_hit_rate)

opt_input_tokens  = queries_needing_llm * AVG_INPUT_TOKENS
opt_output_tokens = queries_needing_llm * AVG_OUTPUT_TOKENS
opt_llm_input     = (opt_input_tokens  / 1_000_000) * INPUT_COST_PER_1M
opt_llm_output    = (opt_output_tokens / 1_000_000) * OUTPUT_COST_PER_1M
optimized_llm_cost = opt_llm_input + opt_llm_output
optimized_storage  = storage_cost * 1.2   # 20% overhead for cache index
total_cost_optimized = optimized_llm_cost + embedding_cost + optimized_storage

print(f"\n✨ Cost Breakdown (WITH Optimization — {cag_hit_rate * 100:.1f}% cache hit rate):")
print(f"   Queries cached   : {queries_hitting_cache:,.0f}")
print(f"   Queries to LLM   : {queries_needing_llm:,.0f}")
print(f"   LLM Generation   : ${optimized_llm_cost:.2f} (saved ${total_llm_cost - optimized_llm_cost:.2f})")
print(f"   Embeddings       : ${embedding_cost:.2f}")
print(f"   Storage + cache  : ${optimized_storage:.2f}")
print(f"   ─────────────────────")
print(f"   TOTAL            : ${total_cost_optimized:.2f}/month")

monthly_savings = total_cost_baseline - total_cost_optimized
savings_percent = (monthly_savings / total_cost_baseline) * 100 if total_cost_baseline > 0 else 0
annual_savings  = monthly_savings * 12

print(f"\n💵 Cost Savings:")
print(f"   Monthly : ${monthly_savings:.2f} ({savings_percent:.1f}% reduction)")
print(f"   Annual  : ${annual_savings:.2f}")

# ── ROI Analysis ──────────────────────────────────────────────
total_implementation = (DEV_HOURS * HOURLY_RATE) + SETUP_COST
payback_months       = total_implementation / monthly_savings if monthly_savings > 0 else float("inf")
roi_12mo             = (annual_savings - total_implementation) / total_implementation * 100

print(f"\n📈 ROI Analysis:")
print(f"   Implementation Cost : ${total_implementation:,.0f}")
print(f"      Development      : ${DEV_HOURS * HOURLY_RATE:,.0f} ({DEV_HOURS}h @ ${HOURLY_RATE}/h)")
print(f"      Setup            : ${SETUP_COST:,.0f}")
print(f"   ─────────────────────")
print(f"   Payback Period      : {payback_months:.1f} months")
print(f"   12-month ROI        : {roi_12mo:.1f}%")

# ── Scaling Scenarios ─────────────────────────────────────────
print(f"\n📊 Scaling Scenarios:")
print(f"   {'Users':<10} {'Queries/mo':<15} {'Baseline':<12} {'Optimized':<12} {'Savings':<12}")
print(f"   {'-'*10} {'-'*15} {'-'*12} {'-'*12} {'-'*12}")
for factor in [1, 2, 5, 10]:
    users     = DAILY_USERS * factor
    queries   = MONTHLY_QUERIES * factor
    baseline  = total_cost_baseline * factor
    optimized = total_cost_optimized * factor
    savings   = baseline - optimized
    print(f"   {users:<10,} {queries:<15,} ${baseline:<11,.0f} ${optimized:<11,.0f} ${savings:<11,.0f}")

# Save JSON
cost_analysis = {
    "assumptions": {
        "daily_users":        DAILY_USERS,
        "queries_per_user":   QUERIES_PER_USER,
        "monthly_queries":    MONTHLY_QUERIES,
        "avg_input_tokens":   AVG_INPUT_TOKENS,
        "avg_output_tokens":  AVG_OUTPUT_TOKENS,
    },
    "baseline_costs": {
        "llm_generation":  round(total_llm_cost, 2),
        "embeddings":      round(embedding_cost, 2),
        "storage":         round(storage_cost, 2),
        "total_monthly":   round(total_cost_baseline, 2),
    },
    "optimized_costs": {
        "cache_hit_rate_percent": round(cag_hit_rate * 100, 1),
        "queries_cached":         int(queries_hitting_cache),
        "llm_generation":         round(optimized_llm_cost, 2),
        "embeddings":             round(embedding_cost, 2),
        "storage_with_cache":     round(optimized_storage, 2),
        "total_monthly":          round(total_cost_optimized, 2),
    },
    "savings": {
        "monthly_usd":     round(monthly_savings, 2),
        "annual_usd":      round(annual_savings, 2),
        "savings_percent": round(savings_percent, 1),
    },
    "roi": {
        "implementation_cost":    total_implementation,
        "payback_months":         round(payback_months, 1),
        "twelve_month_roi_percent": round(roi_12mo, 1),
    },
}

output_path = output_dir / "cost_analysis.json"
try:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cost_analysis, f, indent=2)
    print(f"\n💾 Saved cost analysis to: {output_path}")
except Exception as e:
    print(f"❌ Failed to save cost_analysis.json: {e}")

### 6b. Break-Even & Scale Sensitivity

Shows the minimum daily users for 12-month payback and how savings grow at scale.

In [ ]:
print("=" * 80)
print("BREAK-EVEN & SCALE SENSITIVITY")
print("=" * 80)

saving_per_query  = cost_per_query_usd * cag_hit_rate
queries_for_be    = total_implementation / (saving_per_query * 12) if saving_per_query > 0 else float("inf")
users_for_be      = queries_for_be / (QUERIES_PER_USER * 30)

print(f"\nCurrent Pilot ({DAILY_USERS} daily users):")
print(f"  Monthly LLM cost (no cache) : ${total_cost_baseline:.2f}")
print(f"  Monthly savings from CAG    : ${monthly_savings:.2f}")
print(f"  Payback period              : {payback_months:.0f} months")
print(f"  Primary value at pilot scale: latency ({speedup:.0f}x faster), not cost.")

print(f"\nBreak-Even Target (12-month payback):")
print(f"  Saving per cached query : ${saving_per_query:.6f}")
print(f"  Monthly queries needed  : {queries_for_be:,.0f}")
print(f"  Daily users needed      : {users_for_be:,.0f}")

print(f"\nSavings at Production Scales:")
print(f"  {'Daily Users':<14} {'Monthly Queries':<18} {'Monthly Savings':<18} {'Payback (mo)'}")
print(f"  {'-'*66}")
for test_users in [500, 5_000, 50_000, 500_000]:
    mq     = test_users * QUERIES_PER_USER * 30
    ms     = mq * saving_per_query
    pb     = (total_implementation / ms) if ms > 0 else float("inf")
    pb_str = f"{pb:.1f}" if pb < 999 else ">999"
    ok     = " [payback <12mo]" if pb <= 12 else ""
    print(f"  {test_users:<14,} {mq:<18,} ${ms:<17,.2f} {pb_str}{ok}")

print(f"\nKey Takeaway:")
print(f"  CAG delivers strong ROI at 50k+ daily users (payback < 12 months).")
print(f"  At {DAILY_USERS} users the architecture proves correctness and latency gain.")
print(f"  Deploy CAG from day one: scales to positive ROI without re-architecting.")

### 7. Final Summary & Deliverables Check

In [ ]:
print("=" * 80)
print("PART 4: PERFORMANCE ARENA — COMPLETE!")
print("=" * 80)

# Rubric-required output files
required_files = [
    ("chunking_comparison.csv", "Chunking Strategy Comparison (8 pts)"),
    ("cag_stats.json",          "CAG Cache Effectiveness (6 pts)"),
    ("crag_impact.csv",         "CRAG Correction Impact (6 pts)"),
    ("cost_analysis.json",      "BONUS: Cost Analysis (+5 pts)"),
]

print("\n📋 Deliverables Check:")
all_present = True
for filename, description in required_files:
    filepath = output_dir / filename
    if filepath.exists():
        size_kb = filepath.stat().st_size / 1024
        status  = "✅" if size_kb > 0 else "⚠️  EMPTY"
        print(f"   {status} {filename:30} ({size_kb:.1f} KB) — {description}")
        if size_kb == 0:
            all_present = False
    else:
        print(f"   ❌ {filename:30} MISSING — {description}")
        all_present = False

# Additional column check for chunking CSV
chunking_path = output_dir / "chunking_comparison.csv"
if chunking_path.exists():
    try:
        _df = pd.read_csv(chunking_path)
        required_cols = {"precision_at_5", "recall_at_5", "answer_relevance", "latency_ms"}
        missing_cols  = required_cols - set(_df.columns)
        if missing_cols:
            print(f"\n   ⚠️  chunking_comparison.csv missing rubric columns: {missing_cols}")
            all_present = False
        else:
            print(f"\n   ✅ chunking_comparison.csv has all 4 required metric columns.")
    except Exception as e:
        print(f"\n   ⚠️  Could not validate chunking_comparison.csv columns: {e}")

print("\n" + "=" * 80)
if all_present:
    print("🎉 ALL DELIVERABLES GENERATED SUCCESSFULLY!")
    print("=" * 80)
    print("\n📊 Key Findings:")
    print(f"   • Best Chunking Strategy : {winner}")
    print(f"   • CAG Cache Hit Rate     : {hit_rate:.1f}%")
    print(f"   • CAG Speedup            : {speedup:.1f}x")
    print(f"   • CRAG Correction Rate   : {correction_rate:.1f}%")
    print(f"   • Monthly Cost Savings   : ${monthly_savings:.2f}")
    print(f"   • ROI Payback Period     : {payback_months:.1f} months")
    print("\n🏆 Ready for submission!")
else:
    print("⚠️  SOME DELIVERABLES MISSING OR EMPTY — review errors above before submitting.")

print("=" * 80)

# Final assert so the notebook exits non-zero if files are missing
assert all_present, (
    "❌ One or more required output files are missing or empty. "
    "Re-run the relevant sections before submission."
)